# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ilhamilha-creator/flyrank-ml-assignments/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

- Unit of Analysis (Grain): One unique content page (content_id).
- Time Window: Each row aggregates search performance over a 90-day historical window, with the target label (trend_direction) indicating performance trajectory over that same period.

In [ ]:
import pandas as pd

# Load the starter dataset to verify the grain
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

print("Grain Verification:")
print(f"Unit of analysis: One unique content page (content_id)")
print(f"Total rows: {len(df):,}")
print(f"Unique content_ids: {df['content_id'].nunique():,}")

# Verify the grain - check for duplicates
duplicates = df[df.duplicated(subset=['content_id'], keep=False)]
print(f"Duplicate content_ids: {len(duplicates)} (should be 0)")

if len(duplicates) == 0:
    print("✓ Grain verified: Each row = one unique content page")
else:
    print("✗ Grain issue found: Duplicate content_ids exist")

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

- Features (Predictive signals available at decision moment):
  1. content_age_days - How old the content is (knowable before refresh decision)
  2. days_since_last_update - Freshness signal (knowable before refresh decision)
  3. impressions_90d - Search visibility volume (historical performance)
  4. ctr - Click-through rate (engagement quality signal)
  5. avg_position - Average search ranking position (visibility quality)
  6. word_count - Content length indicator (effort required for refresh)
  7. search_volume - Keyword demand indicator (opportunity size)

- Label / Target: trend_direction - Categorical target ('down' = declining, needs refresh; 'up'/'stable' = healthy)

- Context Fields: content_id, client_id - For grouping and joining, not for model learning

- Excluded Fields: 
  1. trend_pct - Excluded because it's directly derived from the label (leakage risk)
  2. Any product decision flags - Not present in dataset (intentionally excluded)
  3. Pages with impressions_90d = 0 - Excluded because no visibility means no measurable decline pattern

In [ ]:
# Show the feature structure from the actual dataset
print("Feature Structure Verification:")
print("\nKey features available:")
key_features = ['content_age_days', 'days_since_last_update', 'impressions_90d', 'ctr', 'avg_position', 'word_count', 'search_volume']
for feat in key_features:
    if feat in df.columns:
        null_count = df[feat].isnull().sum()
        print(f"  {feat}: {null_count} null values ({null_count/len(df)*100:.1f}%)")

print(f"\nTarget label: trend_direction")
print(f"  Distribution: {df['trend_direction'].value_counts().to_dict()}")

print(f"\nContext fields: content_id, client_id")
print(f"  Unique content_ids: {df['content_id'].nunique():,}")
print(f"  Unique client_ids: {df['client_id'].nunique():,}")

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Verification Claims Plan:
- Query 1: Prove grain uniqueness - verify that content_id is truly unique per row
- Query 2: Extract row counts and basic statistics to verify data completeness
- Query 3: Audit missing data and verify availability with IS TRUE filters

In [ ]:
print("--- Query 1: Verify Grain Uniqueness ---")
duplicate_check = df.groupby('content_id').size().reset_index(name='count')
duplicates = duplicate_check[duplicate_check['count'] > 1]
print(f"Duplicate content_ids found: {len(duplicates)} (should be 0)")
if len(duplicates) == 0:
    print("✓ Grain uniqueness verified")
else:
    print("✗ Grain issue: duplicates exist")

print("\n--- Query 2: Row Counts & Basic Statistics ---")
print(f"Total rows: {len(df):,}")
print(f"Total columns: {len(df.columns)}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

print("\n--- Query 3: Availability & Null Audit ---")
print("Key feature availability:")
for feat in key_features:
    available = df[feat].notna().sum()
    total = len(df)
    availability_pct = (available / total) * 100
    print(f"  {feat}: {available:,}/{total:,} ({availability_pct:.1f}%) available")

print("\n--- Query 4: IS TRUE Filter Test ---")
# Test how many rows survive basic availability filters
filtered_df = df[
    (df['impressions_90d'] > 0) & 
    (df['content_age_days'] >= 90) &
    (df['ctr'].notna()) &
    (df['avg_position'].notna())
]
print(f"Rows after IS TRUE filters: {len(filtered_df):,} / {len(df):,} ({len(filtered_df)/len(df)*100:.1f}%)")

## 4. Five features, max

*Build a small feature frame for your lane. Give every feature one line: "knowable at the decision moment because..."*

Five core features for Refresh / Content Opportunity Scoring:

1. content_age_days - Knowable at decision moment because content creation date is historical metadata available before any refresh decision
2. days_since_last_update - Knowable at decision moment because last update timestamp is content history available before deciding to refresh
3. impressions_90d - Knowable at decision moment because it's historical search performance over the past 90 days, measured before the refresh decision
4. ctr - Knowable at decision moment because it's calculated from historical clicks/impressions over the 90-day window prior to decision
5. avg_position - Knowable at decision moment because it's the average search ranking position over the historical 90-day period

In [ ]:
# Build the feature frame with the five core features
feature_frame = df[[
    'content_age_days', 
    'days_since_last_update', 
    'impressions_90d', 
    'ctr', 
    'avg_position',
    'trend_direction'  # target
]].copy()

print("Feature Frame Dimensions:")
print(f"Rows: {len(feature_frame):,}")
print(f"Features: {len(feature_frame.columns) - 1}")  # -1 for target

print("\nFeature Statistics:")
print(feature_frame.describe())

print("\nTarget Distribution:")
print(feature_frame['trend_direction'].value_counts())

## 5. The trap: deliberate leakage, then delete it

*Add ONE label-derived column on purpose, watch your quick score jump toward perfect, then delete it and keep the honest number.*

In [ ]:
# THE LEAKAGE TRAP: Adding trend_pct (label-derived feature)
print("=== LEAKAGE EXPERIMENT ===")
print("Adding trend_pct which is directly derived from trend_direction (the label)")

# Create a version with the leaky feature
feature_frame_leaky = feature_frame.copy()
feature_frame_leaky['trend_pct'] = df['trend_pct']  # This is derived from the label!

print(f"\nWith leaky feature (trend_pct):")
print(f"Correlation with target: {feature_frame_leaky['trend_pct'].abs().corr((feature_frame_leaky['trend_direction'] == 'down').astype(int)):.3f}")
print("This near-perfect correlation shows why trend_pct leaks the label")

# Simple baseline model to show the effect
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score

X_leaky = feature_frame_leaky[['content_age_days', 'days_since_last_update', 'impressions_90d', 'ctr', 'avg_position', 'trend_pct']].fillna(0)
X_honest = feature_frame[['content_age_days', 'days_since_last_update', 'impressions_90d', 'ctr', 'avg_position']].fillna(0)
y = (feature_frame['trend_direction'] == 'down').astype(int)

# Leaky model
leaky_model = DecisionTreeClassifier(max_depth=2, random_state=42)
leaky_scores = cross_val_score(leaky_model, X_leaky, y, cv=3, scoring='accuracy')

# Honest model  
honest_model = DecisionTreeClassifier(max_depth=2, random_state=42)
honest_scores = cross_val_score(honest_model, X_honest, y, cv=3, scoring='accuracy')

print(f"\nModel Accuracy with LEAKY feature: {leaky_scores.mean():.3f}")
print(f"Model Accuracy with HONEST features: {honest_scores.mean():.3f}")
print(f"\nLeakage inflates accuracy by: {(leaky_scores.mean() - honest_scores.mean()):.3f}")

print("\n=== DELETING LEAKY FEATURE ===")
print("Keeping only honest features for real modeling")
print("Honest features: content_age_days, days_since_last_update, impressions_90d, ctr, avg_position")

## 6. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Named Limitations & Boundaries:

1. **Time Window Limitation**: The 90-day aggregation window cannot capture longer-term seasonal patterns or year-over-year trends. It shows recent performance but not historical context.

2. **Proxy Label Limitation**: trend_direction is a current-state proxy, not a future outcome. It doesn't guarantee that refreshing a page will improve future performance - it only identifies current decline patterns.

3. **Single Channel Data**: This dataset only contains organic search performance (GSC data). It cannot capture user behavior from other channels (direct, social, referral) or multi-touch attribution patterns.

4. **No Causal Claims**: The data shows correlations and patterns but cannot prove causation about Google's algorithm or guarantee that specific refresh actions will cause ranking improvements.

5. **Static Snapshot**: This is a cross-sectional snapshot, not time-series data. We cannot model temporal dynamics or how performance changes over time for individual pages.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

- Unit of Analysis (Grain): A unique combination of a specific Report Date, a Pseudonymized Client ID, and a Pseudonymized URL string instance (date + client_hash_id + content_hash_id).
- Time Window: We are targeting a stable mid-panel operational month from historical data, specifically March 2026 (month=2026-03), leaving the final month as a clean, sealed evaluation window.

In [21]:
import os
import duckdb
from google.colab import userdata
from google.colab.userdata import SecretNotFoundError

hf_token = None
con = None
rel = None

try:
    hf_token = userdata.get('HF_TOKEN')
except SecretNotFoundError:
    print("Error: The 'HF_TOKEN' secret is not found. Please ensure it is set in Colab secrets.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

if hf_token:
    con = duckdb.connect()
    con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
    rel = "hf://datasets/FlyRank/internship-warehouse"

    # FIXED: Changed date to report_date, impressions to gsc_impressions, clicks to gsc_clicks
    query_grain = f"""
        SELECT report_date, client_hash_id, content_hash_id, gsc_impressions, gsc_clicks
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
        LIMIT 3
    """
    print(con.sql(query_grain).df().to_string())
else:
    print("Hugging Face token is not available.")

  report_date           client_hash_id           content_hash_id  gsc_impressions  gsc_clicks
0  2026-03-01  client_73cda7b4e4f265ea  content_b7e512995f79d5a6               20           0
1  2026-03-01  client_73cda7b4e4f265ea  content_05597932fe4da067                1           0
2  2026-03-01  client_73cda7b4e4f265ea  content_7a105f548d9c6916              125           1


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

- Features (Predictive signals available at decision moment):
  1. feat_current_impressions (raw daily volume)
  2. feat_current_clicks (historical intent match)
  3. feat_current_position (search ranking visibility placement tier)
  4. feat_current_ctr (computed conversion velocity proxy)
  5. feat_is_top_3 (binary indicator profile flag)
- Label / Target: trend_direction (Binary categorical flag indicating whether performance is decaying 'down' or growing 'up').
- Context Fields: date, client_hash_id, content_hash_id.
- Excluded Fields: Pages generating exactly 0 impressions over the entire 90-day baseline panel are intentionally excluded to eliminate extreme cold-start noise and empty search queries.

In [25]:
if con and rel:
    # Définition complète de la requête avec toutes les variables (Features)
    query_fields = f"""
        SELECT
            report_date, client_hash_id, content_hash_id,
            gsc_impressions as feat_current_impressions,
            gsc_clicks as feat_current_clicks,
            gsc_avg_position as feat_current_position,
            (gsc_clicks::FLOAT / NULLIF(gsc_impressions, 0)) as feat_current_ctr,
            CASE WHEN gsc_avg_position <= 3 THEN 1 ELSE 0 END as feat_is_top_3,
            -- THE LEAKAGE TRAP
            LEAD(gsc_clicks, 1) OVER (PARTITION BY content_hash_id ORDER BY report_date) as LEAK_future_clicks_trap
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_avg_position IS NOT NULL
        LIMIT 5
    """
    # Exécution et affichage du tableau de résultats
    print(con.sql(query_fields).df().to_string())
else:
    print("Cannot execute query: Setup cell failed.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  report_date           client_hash_id           content_hash_id  feat_current_impressions  feat_current_clicks  feat_current_position  feat_current_ctr  feat_is_top_3  LEAK_future_clicks_trap
0  2026-03-01  client_157ffe4d4a595515  content_00039f4c7a954114                         6                    0               7.833333               0.0              0                        0
1  2026-03-02  client_157ffe4d4a595515  content_00039f4c7a954114                         1                    0               8.000000               0.0              0                        0
2  2026-03-03  client_157ffe4d4a595515  content_00039f4c7a954114                         9                    0               6.333333               0.0              0                        0
3  2026-03-04  client_157ffe4d4a595515  content_00039f4c7a954114                         6                    0               5.666667               0.0              0                        0
4  2026-03-05  client_157ffe4d4a595

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Verification Claims Plan:
- Query 1: Prove unique constraints to verify that the specified composite keys truly establish the exact grain of the warehouse.
- Query 2: Extract total active row counts and bounding dates to map out data completeness for March 2026.
- Query 3: Audit missing data fields and test data availability profiles using strict boolean constraints.

In [27]:
if con and rel:
    print("--- Query 1: Verify Grain Uniqueness ---")
    q1 = f"""
        SELECT report_date, client_hash_id, content_hash_id, COUNT(*)
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
        GROUP BY report_date, client_hash_id, content_hash_id
        HAVING COUNT(*) > 1
        LIMIT 5
    """
    print(f"Duplicate rows found at grain: {len(con.sql(q1).df())}")

    print("\n--- Query 2: Row Counts & Full Date Span ---")
    q2 = f"""
        SELECT COUNT(*) as total_rows, MIN(report_date) as earliest_logged, MAX(report_date) as latest_logged
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    """
    print(con.sql(q2).df().to_string())

    print("\n--- Query 3: Availability & Null Audit ---")
    q3 = f"""
        SELECT
            COUNT(*) as total,
            COUNT(gsc_avg_position) as non_null_positions,
            COUNT(gsc_clicks) as non_null_clicks
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    """
    print(con.sql(q3).df().to_string())
else:
    print("Cannot execute verification queries.")

--- Query 1: Verify Grain Uniqueness ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate rows found at grain: 0

--- Query 2: Row Counts & Full Date Span ---
   total_rows earliest_logged latest_logged
0     9841378      2026-03-01    2026-03-31

--- Query 3: Availability & Null Audit ---
     total  non_null_positions  non_null_clicks
0  9841378             3611061          9841378


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Named Limitations & Boundaries:
1. Short-Horizon Bias: March 2026 data reflects a brief localized time slice. It cannot catch seasonal macroeconomic shifts or core engine structural update movements.
2. Unbalanced Depth: Historical data depth varies significantly by enterprise client as outlined in the client dimension files, creating inconsistent baseline history depth.
3. System Limits: This panel relies entirely on historical GSC metrics and cannot profile multi-channel user behaviors occurring outside organic search footprints.

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Contract verification steps finalized successfully. Data pipelines match rules.")

Contract verification steps finalized successfully. Data pipelines match rules.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.